![header](img/header.PNG)
# Best practices to download GloFAS data from CEMS Early Warning Data Store
## 4th CEMS Global Flood meeting, Interactive session 2: 
### Part 2 : Best practices to download GloFAS data from CEMS Early Warning Data Store 

Welcome to the companion Jupyter notebook of the 4th CEMS Global Flood meeting -  Interactive session 2 (part 2): Best practices to download GloFAS data from CEMS Early Warning Data Store. 

This notebook will walk you through the best practices to download glofas datasets:
- GloFAS Climatology
- GloFAS Forecast
- GloFAS Reforecast
- GloFAS Seasonal forecast
- GloFAS Seasonal Reforecast

<div style="padding: 20px; background-color: #D4E5F7; border-left: 6px solid #006EAD; margin-bottom: 15px; width: 95%;">
</div>

The EWDS retrieval times can vary significantly depending on the number of requests that the EWDS has at any one time and also based on the following factors that affect GloFAS:
* The priority of the dataset in question
* The size of the request
* The number of requests submitted by a user
* The number of requests to retrieve data from ECMWF Archive
* The number of requests requesting a specific dataset
* The size of the overall queue.

The EWDS delivers data as fast as possible, however, it is not an operational service and should not be relied upon to deliver data in real-time as it is produced.



In [ ]:
import os
import glob
# CDS API
import cdsapi

# Libraries for working with multidimensional arrays
import numpy as np
import xarray as xr

import datetime
#Progress bar
from tqdm import tqdm

In [ ]:
DATADIR = '/scratch/ecm3644/glofas-workshop/Last-test-download/'
os.makedirs(DATADIR, exist_ok=True)
HOME_DIR = os.path.expanduser("~")

## Retrieve GloFAS Climatology
### Best practice : Loop over years

In [ ]:
c = cdsapi.Client()

DATASET = 'cems-glofas-historical'
YEARS = ['%02d' % (mn) for mn in range(1979, 2023)]
MONTHS = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
DAYS = ['%02d' % (mn) for mn in range(1, 32)]

for year in tqdm(YEARS, desc="Downloading data", unit="year"):
    REQUEST = {
        'system_version': 'version_4_0',
        'product_type': 'consolidated',
        'hydrological_model': 'lisflood',
        'variable': 'river_discharge_in_the_last_24_hours',
        'hyear': year,
        'hmonth': MONTHS,
        'hday': DAYS,
        'data_format': "grib",
        'download_format': "zip"
    }
    # Download file for the given year
    c.retrieve(DATASET, REQUEST).download(f'{DATADIR}{DATASET}_{year}.zip')

## Retrieve GloFAS Forecast
### Best practice : Loop over years, months, days

In [ ]:
# Function to generate date list
def compute_dates_range(start_date,end_date,loop_days=True):
    start_date = datetime.date(*[int(x) for x in start_date.split('-')])
    end_date = datetime.date(*[int(x) for x in end_date.split('-')])
    ndays =  (end_date - start_date).days + 1
    dates = []
    for d in range(ndays):
        dates.append(start_date + datetime.timedelta(d))
    if not loop_days:
        dates = [i for i in dates if i.day == 1]
    else:
        pass
    return dates


# Initialize CDS API client
c = cdsapi.Client()

# User inputs
DATASET = 'cems-glofas-forecast'
START_DATE = '2024-05-26'
END_DATE = '2024-10-01'
LEADTIMES = [str(lt) for lt in range(24, 744, 24)]

# Generate date list
dates = compute_dates_range(START_DATE, END_DATE)

# loop over dates and save to disk
for date in tqdm(dates, desc="Downloading data"):
    year, month, day = date.strftime('%Y'), date.strftime('%m'), date.strftime('%d')
    print(f"Retrieving: {year}-{month}-{day}")

    REQUEST = {
        'system_version': 'operational',
        'hydrological_model': 'lisflood',
        'product_type': 'control_forecast',
        'variable': 'river_discharge_in_the_last_24_hours',
        'year': year,
        'month': month,
        'day': day,
        'leadtime_hour': LEADTIMES,
        'data_format': "grib2",
        'download_format': "zip"
    }
    c.retrieve(DATASET, REQUEST).download(f'{DATASET}_{year}_{month}_{day}.zip')

## Retrieve GloFAS reforecast
### Best practice: Loop over months and days. Make your area of interest precise.

#### The reforecast has been produced every Monday and Thursday of each week until the release of the new IFS cycle 49r1 (13 Nov 2024). The medium-range reforecast is now run for fixed days of the month: 1/5/9/13/17/21/25/29 (excluding 29 February).

<div style="padding: 20px; background-color: #D4E5F7; border-left: 6px solid #006EAD; margin-bottom: 15px; width: 95%;">
    <strong>Case 1 </strong>: Before cycle 49r1 
</div>

In [ ]:
## === retrieve GloFAS Reforecast ===
   
## === subset India, Pakistan, Nepal and Bangladesh region ===
import cdsapi
from datetime import datetime, timedelta
def get_monthsdays():
    start, end = datetime(2024, 1, 1), datetime(2024,1, 31) # reference year 2024
    days = [start + timedelta(days=i) for i in range((end - start).days + 1)]
    monthday = [d.strftime("%m-%d").split("-") for d in days if d.weekday() in [0, 3]]
   
    return monthday
MONTHSDAYS = get_monthsdays()
c = cdsapi.Client()    
# user inputs
DATASET='cems-glofas-reforecast'
BBOX = [35 ,-5, 30, 5] # North West South East
YEARS  = ['%d'%(y) for y in range(2022,2023)]
LEADTIMES = ['%d'%(l) for l in range(24,1128,24)]  
# submit request
for md in MONTHSDAYS:
    month = md[0].lower()
    day = md[1]
   
    REQUEST= {
            'system_version': ["version_4_0"],
            'variable': 'river_discharge_in_the_last_24_hours',
            'hydrological_model': 'lisflood',
            'product_type': 'control_reforecast',
            'area': BBOX,# < - subset
            'hyear': YEARS,
            'hmonth': month,
            'hday': day,
            'leadtime_hour': LEADTIMES,
            'data_format': "grib2",
            'download_format': "zip"
             }
    c.retrieve(DATASET, REQUEST).download(f'{DATASET}_{month}_{day}.zip')

<div style="padding: 20px; background-color: #D4E5F7; border-left: 6px solid #006EAD; margin-bottom: 15px; width: 95%;">
    <strong>Case 2 </strong>: Cycle 49r1 
</div>
<div style="padding: 20px; background-color: #D4E5F7; border-left: 6px solid #006EAD; margin-bottom: 15px; width: 95%;">
    <strong>Important </strong>Temporary freeze of medium-range reforecasts available from the EWDS, with no update of the dataset from GloFAS v4.2 release date. If you require access to the reforecast data, please get in touch with us via the ECMWF Support Portal using the title 'Access to GloFAS Reforecast Data' when creating the ticket.
</div>

In [ ]:
c = cdsapi.Client()

# User inputs
DATASET = 'cems-glofas-reforecast'
BBOX = [35, -5, 30, 5]  # North West South East
YEARS = [str(y) for y in range(2020, 2025)]  # Last 20 years
LEADTIMES = [str(l) for l in range(24, 1128, 24)]  # Forecast lead times

# Define the target period (e.g., January 1 to January 17, 2025)
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 1, 17)

target_days_list = [1, 5, 9, 13, 17, 21, 25, 29]

# Generate list of (month, day) for the period, only if day is in target_days_list
target_days = []
current_date = start_date
while current_date <= end_date:
    if int(current_date.strftime('%d')) in target_days_list:  # Check if day is in the target list
        target_days.append((current_date.strftime('%m'), current_date.strftime('%d')))
    current_date += timedelta(days=1)

# Loop through years and target days
for year in tqdm(YEARS, desc="Downloading reforecast data"):
    for month, day in target_days:
        if month == "02" and day == "29":  # Skip Feb 29 for non-leap years
            continue

        REQUEST = {
            'system_version': ["version_4_0"],
            'variable': 'river_discharge_in_the_last_24_hours',
            'hydrological_model': 'lisflood',
            'product_type': 'control_reforecast',
            'area': BBOX,  # < - subset
            'hyear': year,  # Historical year
            'hmonth': month,
            'hday': day,
            'leadtime_hour': LEADTIMES,
            'data_format': "grib2",
            'download_format': "zip"
        }

        print(f"Downloading: {DATASET} for {year}-{month}-{day}")
        c.retrieve(DATASET, REQUEST).download(f'{DATADIR}{DATASET}_{year}_{month}_{day}.zip')

## Retrieve GloFAS Seasonal
### Best practice : Loop over years and months. Make your area of interest precise.

In [ ]:
## === Retrieve GloFAS Seasonal Forecast ===
 
c = cdsapi.Client()
     
# user inputs
DATASET = 'cems-glofas-seasonal'
     
YEARS = ['%d' % (y) for y in range(2022, 2023)]
MONTHS = ['%02d' % (m) for m in range(1, 13)]
LEADTIMES = ['%d' % (l) for l in range(24, 2976, 24)]
 
for year in YEARS:
    print(f'year_{year}')
    for month in MONTHS:
        print(f'Month_{month}')
        REQUEST = {
            'system_version': ['operational'],
            "hydrological_model": ["lisflood"],
            'variable': 'river_discharge_in_the_last_24_hours',
            'year': year,
            'month': '12' if year == '2020' else month,
            'leadtime_hour': LEADTIMES,
            'area': [90, -180, -90, 180],
            'data_format': 'grib2',
            'download_format': 'unarchived'
        }
        c.retrieve(DATASET, REQUEST).download(f'{DATASET}_{year}_{month}.grib')

## Retrieve GloFAS Seasonal Reforecast
### Best practice : Loop over years and months. Make your area of interest precise.

In [ ]:
## === Retrieve GloFAS Seasonal Reforecast ===
   
## === subset South America/Amazon region ===
   
import cdsapi
   
c = cdsapi.Client()
     
# user inputs
DATASET='cems-glofas-seasonal-reforecast'
   
YEARS  = ['%d'%(y) for y in range(1981,2021)]
   
MONTHS = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
   
LEADTIMES = ['%d'%(l) for l in range(24,2976,24)]
       
for year in YEARS:
    for month in MONTHS:
        REQUEST={
                'system_version': 'version_4_0',
                'variable':'river_discharge_in_the_last_24_hours',
                'hydrological_model':'lisflood',
                'hyear': year,
                'hmonth': month,
                'leadtime_hour': LEADTIMES,
                'area': [ 10.95, -90.95, -30.95, -29.95 ],
                'data_format': 'netcdf',
                'download_format': 'unarchived'
                }
        c.retrieve(DATASET, REQUEST).download(f'{DATASET}_{year}_{month}.netcdf')